In [1]:
# import des librairie utiles
import requests
from pprint import pprint
import pandas as pd
import numpy as np
import threading
from tqdm import tqdm
from typing import Callable
import time


In [11]:
df_final = pd.read_csv('C:\\Users\\const\\OneDrive\\Documents\\Formation Data\\Projet 2\\Sprint 4\\Demo_projet2\\data\\df_final.csv')
df_final = df_final.dropna(subset=['overview','production_countries']) # on drop les ligne qui contiennent des valeurs manquante dans les colonnes genres, production_countries et keywords, on en a pas besoin pour la suite du projet

In [30]:
df_ml = df_final[['genres','production_countries','id']] # on selectionne les colonnes qui nous interesse pour la suite du projet

In [31]:
df_ml = df_ml[df_ml['production_countries'].isin(['US','FR'])] # on selectionne les film qui ont été produit aux US ou en FR pour la suite du projet

In [32]:
df_ml.head()

,genres,production_countries,id
0,"['Animation', 'Comédie', 'Aventure', 'Familial']",US,57800
1,"['Aventure', 'Animation', 'Comédie', 'Familial...",US,11688
2,"['Fantastique', 'Aventure', 'Action']",US,284052
3,"['Science-Fiction', 'Action', 'Aventure']",US,1724
6,"['Action', 'Thriller', 'Drame']",US,1119878


In [52]:
print(type(df_ml['genres'].iloc[0]))

<class 'str'>


In [33]:
df_ml['genres'] = df_ml['genres'].apply(lambda row: row.replace('[','').replace(']','').replace("'","").replace("'","").replace(' ',''))

def filtrer_genres(genres):
    liste = genres.split(',')
    if 'Animation' in liste and 'Familial' in liste:
        return ','.join([g for g in liste if g in ['Animation', 'Familial']])
    return genres

df_ml['genres'] = df_ml['genres'].apply(filtrer_genres)
df_genre = df_ml['genres'].str.get_dummies(sep=',') # on transforme la colonne genre en liste de genre, idem pour les pays de production
df_ml['production_countries'] = df_ml['production_countries'].map({'US': 0, 'FR': 1}) # on transforme la colonne production_countries en binaire, 1 pour les film produit aux US et 0 pour les film produit en FR
df_ml = pd.concat([df_ml, df_genre], axis=1) # on concatene les deux data frame pour n'en faire qu'un seul
display(df_ml.head(3))


,genres,production_countries,id,Action,Animation,Aventure,Comédie,Crime,Documentaire,Drame,...,Guerre,Histoire,Horreur,Musique,Mystère,Romance,Science-Fiction,Thriller,Téléfilm,Western
0,"Animation,Familial",0,57800,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,"Animation,Familial",0,11688,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,"Fantastique,Aventure,Action",0,284052,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import NearestNeighbors

scaler = StandardScaler()
minmax_scaler = MinMaxScaler()



In [34]:
df_ml_test = df_ml.drop(columns=['genres']) # on drop les colonnes qui contiennent des données non numériques pour la suite du projet

In [ ]:
# X = scaler.fit_transform(df_ml_test.select_dtypes(include='number')) # on selectionne les colonnes qui contiennent des données numériques pour les normaliser

In [ ]:
# Filtrer les films qui ont au moins Action ou Comédie
mask = (df_ml_test['Action'] == 1) | (df_ml_test['Comédie'] == 1)
df_ml_filtered = df_ml_test[mask].reset_index(drop=True)

In [9]:
df_ml_test.head()

,vote_count,Action,Animation,Aventure,Comédie,Crime,Documentaire,Drame,Familial,Fantastique,Guerre,Histoire,Horreur,Musique,Mystère,Romance,Science-Fiction,Thriller,Téléfilm,Western
0,8068,0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,7258,0,1,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
2,23352,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,12626,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,17506,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0


In [35]:
df_ml_test.to_csv('df_reco_animefiltré.csv', index=False)

In [ ]:
df_ml_test = df_ml.drop(columns=['genres', 'keywords','release_date'])

In [ ]:
df_reco = df_ml_filtered

In [ ]:
# on préparte le features 
cols_num = ['release_year','runtime', 'vote_average', 'vote_count'] # on selectionne les colonnes numériques pour la suite du projet
cols_binaires = [col for col in df_reco.columns if col not in cols_num]

X_mun = scaler.fit_transform(df_reco[cols_num])# on normalise les colonnes numériques et augmente leur influences
X_bin = df_reco[cols_binaires].values # on garde les colonnes binaires et on réduit leur influences


In [187]:
X = np.hstack([X_mun, X_bin]) # on concatene les deux matrices pour n'en faire qu'une seule

model = NearestNeighbors(n_neighbors=6, metric='cosine') # on crée un modèle de voisinage pour trouver les films les plus similaires entre eux
model.fit(X)

distances, indices = model.kneighbors(X)

print(distances)
print(indices)

[[2.22044605e-16 6.38853415e-10 7.40401185e-10 7.56440133e-10
  8.09661893e-10 8.45353787e-10]
 [2.22044605e-16 9.45548595e-09 1.07165147e-08 1.20672554e-08
  1.29584289e-08 1.37497213e-08]
 [0.00000000e+00 1.18687282e-11 1.69242398e-11 2.08947304e-11
  2.23077112e-11 2.25984786e-11]
 ...
 [0.00000000e+00 1.54321000e-14 1.57651669e-14 2.30926389e-14
  5.10702591e-14 1.06026299e-13]
 [0.00000000e+00 7.20534743e-14 1.33337785e-13 1.37001521e-13
  1.47215573e-13 1.99174011e-13]
 [1.11022302e-16 1.53665969e-12 2.84761104e-12 3.84103860e-12
  3.93696187e-12 4.87843099e-12]]
[[   0  303  212   37  188  314]
 [   1  105  305  443   90  342]
 [   2   12   53  131  134  107]
 ...
 [5941 5910 5868 5890 5837 5867]
 [5942 5009 5648 5822 5498 5008]
 [5943 5850 5760 5907 5923 5863]]


In [36]:
print("Voisins de la Reine des Neiges 2 (index 135):", indices[135])
print("Voisins de Doctor Strange (index 2):", indices[2])

NameError: name 'indices' is not defined

In [188]:
# Adapter selon df_final qui contient les titres
print(df_final.loc[indices[135], 'title'])
print(df_final.loc[indices[2], 'title'])

135                    La Reine des neiges II
125                            La Proposition
143           Le Labyrinthe : La Terre brûlée
83                                  Send Help
41         Terminator 2 : Le Jugement dernier
216    Manuel de survie à l'apocalypse zombie
Name: title, dtype: object
2                                  Doctor Strange
12     La Planète des singes : Le Nouveau Royaume
53                                       Zootopie
131             Les Aventuriers de l'arche perdue
134              Once Upon a Time... in Hollywood
107                         En eaux très troubles
Name: title, dtype: object


In [ ]:
voisins_ids = df_ml_test.iloc[indices[0][1:]]['id']
# Retrouver les titres dans ton dataset de base
df_final[df_final['id'].isin(voisins_ids)]['title']

(5944, 13063)


In [155]:
def reco_movie(movie:str):
  info_movie = df_final[df_final['title'] == movie]
  indice_movie = info_movie.index.get_loc(info_movie.index[0]) # on trouve l'indice du film dans le dataframe
  reco_indices = indices[indice_movie,1:]
  df_reco = df_final.iloc[reco_indices,:10]
  return df_reco

In [10]:
# on transforme la colonne date en valeur interprétable pour les algorithmes de machine learning, on va extraire l'année de sortie du film
df_ml['release_date'] = pd.to_datetime(df_ml['release_date'], errors='coerce') # on transforme la colonne date en datetime, les valeurs qui ne peuvent pas être transformé seront remplacé par NaT
df_ml['release_year'] = df_ml['release_date'].dt.year # on extrait l'année de sortie du film et on la stock dans une nouvelle colonne


In [ ]:
# Intégratoin keywords
df_ml['keywords'] = df_ml['keywords'].apply(lambda row : row.replace('[','').replace(']','').replace("'","").replace("'","").replace(' ','').replace('"',''))# on transforme la colonne keywords qui est une string en une liste de keywords
# Garder seulement les keywords présents dans au moins 20 films
df_keywords = df_ml['keywords'].str.get_dummies(sep=',')
seuil = 20
df_keywords = df_keywords.loc[:, df_keywords.sum() >= seuil]
df_ml = pd.concat([df_ml, df_keywords], axis=1)

In [156]:
reco_movie('Doctor Strange')

,backdrop_path,id,title,original_title,overview,popularity,poster_path,release_date,vote_average,vote_count
29,/pxL08w7Eq0vKrQ3mJzYp0VHGaKG.jpg,49530,Time Out,In Time,Bienvenue dans un monde où le temps a remplacé...,11.5003,/fJe1P25TNH74vVIMUxyzdv37fMc.jpg,2011-10-27,6.970,12097
307,/2YOnE2qmoyyOUqQsFit12gDSauk.jpg,1242419,Roofman,Roofman,Jeffrey Manchester un vétéran et père en diffi...,12.3814,/p1HTuYJvK9qEeK0gbPijCJ18RQ9.jpg,2025-10-08,7.113,742
213,/hNsYUryiwxcdeTMkaBcPF3iEg0p.jpg,335983,Venom,Venom,"Des symbiotes débarquent sur la Terre, parmi e...",12.8507,/vVusHIRlyyFVS42XnqZso2wGKr.jpg,2018-09-28,6.825,17173
189,/3FO1kbQmRlU1H6Kmn7sTfmd7mw9.jpg,1621,Un fauteuil pour deux,Trading Places,"D'un côté, Louis Winthorpe III, un jeune direc...",16.4162,/kU21ARWABQqoY2a5VRwDxVusT9G.jpg,1983-06-07,7.228,3636
318,/fuacZLpnohQtLD0AhtCErcPVL98.jpg,1259102,Pour l'éternité,Eternity,"Vieux couple marié depuis des années, Joan et ...",12.0617,/c4qswPl8PwnJnyz12m8zVjymWAF.jpg,2025-11-26,7.003,626


In [135]:
reco_movie('Men in Black')

,backdrop_path,id,title,original_title,overview,popularity,poster_path,release_date,vote_average,vote_count
1671,/qQ3aRDQnGmGbtx7OOxKBYyjdLrQ.jpg,10833,Beautés empoisonnées,Heartbreakers,"Arnaqueuses de mère en fille, Max et Page Conn...",4.3590,/ZMLQZkTgmPpxDUf1ITGoxPbZaj.jpg,2001-03-23,6.118,984
1247,/bYSzeJpAHOBMRNJStaubwWwKI72.jpg,207932,Inferno,Inferno,Robert Langdon se réveille dans un hôpital ita...,4.9625,/bCqD2c1nJJY6q65VGBfMItgVvGx.jpg,2016-10-13,6.088,6611
1149,/oK190JudMVb0C5773Vab0TKcBbv.jpg,25642,Ben 10: Alien Swarm,Ben 10: Alien Swarm,"Infectés par des ""nanites"", cellules microscop...",6.4149,/oiwfaErYmlHYRFT6l1UClSX8LqU.jpg,2009-11-15,6.168,376
297,/AecGG1XVCmkk7fT10ko3FC0dLIP.jpg,1043197,Dust Bunny,Dust Bunny,"Aurora, dix ans, demande à un tueur à gages d'...",14.3988,/qaRuV4cbAbrCngM5b7NkGqARDIg.jpg,2025-12-11,6.698,269
1411,/vitMfWMWdOSCqD3DqKQC1h8Cphy.jpg,17927,"Sea, sex & fun",Fired Up!,"Shawn et Nick, les deux ""stars"" de l’école, dé...",5.0694,/livo8ctYaBUekEeVnSKKLKxL53h.jpg,2009-02-20,6.116,688


In [159]:
reco_movie("La Reine des neiges II")

,backdrop_path,id,title,original_title,overview,popularity,poster_path,release_date,vote_average,vote_count
29,/pxL08w7Eq0vKrQ3mJzYp0VHGaKG.jpg,49530,Time Out,In Time,Bienvenue dans un monde où le temps a remplacé...,11.5003,/fJe1P25TNH74vVIMUxyzdv37fMc.jpg,2011-10-27,6.970,12097
307,/2YOnE2qmoyyOUqQsFit12gDSauk.jpg,1242419,Roofman,Roofman,Jeffrey Manchester un vétéran et père en diffi...,12.3814,/p1HTuYJvK9qEeK0gbPijCJ18RQ9.jpg,2025-10-08,7.113,742
213,/hNsYUryiwxcdeTMkaBcPF3iEg0p.jpg,335983,Venom,Venom,"Des symbiotes débarquent sur la Terre, parmi e...",12.8507,/vVusHIRlyyFVS42XnqZso2wGKr.jpg,2018-09-28,6.825,17173
189,/3FO1kbQmRlU1H6Kmn7sTfmd7mw9.jpg,1621,Un fauteuil pour deux,Trading Places,"D'un côté, Louis Winthorpe III, un jeune direc...",16.4162,/kU21ARWABQqoY2a5VRwDxVusT9G.jpg,1983-06-07,7.228,3636
318,/fuacZLpnohQtLD0AhtCErcPVL98.jpg,1259102,Pour l'éternité,Eternity,"Vieux couple marié depuis des années, Joan et ...",12.0617,/c4qswPl8PwnJnyz12m8zVjymWAF.jpg,2025-11-26,7.003,626


In [237]:
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 20)

In [186]:
display(df_final['genres'][df_final['title'] == 'Doctor Strange'])

2    ['Fantastique', 'Aventure', 'Action']
Name: genres, dtype: object

In [149]:
display(df_final['genres'][df_final['title'] == 'La Reine des neiges II'])

135    ['Familial', 'Animation', 'Aventure', 'Comédie', 'Fantastique']
Name: genres, dtype: object

In [30]:
df_first = df_final.sort_values(by='popularity', ascending=False).head(3)

df_first.head()

,backdrop_path,id,title,original_title,overview,popularity,poster_path,release_date,vote_average,vote_count,genres,production_countries,runtime,status,keywords
81,/9Z2uDYXqJrlmePznQQJhL6d92Rq.jpg,1226863,"Super Mario Galaxy, le film",The Super Mario Galaxy Movie,Mario et Luigi gèrent les tracas du royaume Ch...,514.3655,/aSQktALDmbunDbwkuZbZFMEWVFr.jpg,2026-04-01,6.700,714,"['Familial', 'Comédie', 'Aventure', 'Fantastiq...","Japan, US",100,Released,"['galaxy', 'friendship', 'sibling relationship..."
80,/9nzfyiYbmTUXWC4B2kwjl4NAlqO.jpg,1318447,Apex,Apex,Alors qu'elle teste ses limites en solo dans l...,482.8791,/55lEECwWtRWltEa6u3TbiljaoOd.jpg,2026-04-24,6.431,640,"['Action', 'Thriller']","US, Iceland, Australia",96,Released,"['canoe trip', 'rock climbing', 'nutcase', 'su..."
82,/wpabGmGA3IaJNGudbzGritmR8Km.jpg,1007757,Aventures croisées,Swapped,Un petit animal des bois et un oiseau majestue...,311.8469,/gHs3dFVkUdJIuCpCnxUeT1gpole.jpg,2026-05-01,8.200,51,"['Animation', 'Aventure', 'Comédie', 'Familial...","US, Spain",102,Released,"['buddy', 'buddy comedy', 'woodlands', 'loving..."
